###  Notebook 4 - Weather Integration & Advanced Feature Engineering

Integrate historical weather observations with the cleaned traffic dataset.

Weather conditions strongly influence active mobility behaviour. Rainfall, temperature, humidity and wind speed all affect pedestrian, cyclist and scooter movements.

The resulting enhanced dataset will be used for predictive modelling in Notebook 5.

In [1]:
import pandas as pd
import numpy as np
import requests 
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
df = pd.read_csv("../data/processed/master_dataset.csv")

In [3]:
df.head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013


In [4]:
#Brisbane coordinates
LATITUDE = -27.4698
LONGITUDE = 153.0251

In [5]:
url = (
    "https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={LATITUDE}"
    f"&longitude={LONGITUDE}"
    "&start_date=2022-01-01"
    "&end_date=2025-12-31"
    "&daily="
    "temperature_2m_max,"
    "temperature_2m_min,"
    "precipitation_sum,"
    "wind_speed_10m_max,"
    "sunshine_duration"
    "&timezone=Australia/Brisbane"
)

In [6]:
response = requests.get(url)

response.status_code

200

In [7]:
weather_json = response.json()

weather = pd.DataFrame(weather_json["daily"])

weather.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,sunshine_duration
0,2022-01-01,25.5,19.5,14.3,13.3,17634.44
1,2022-01-02,27.5,18.4,2.6,21.1,48883.74
2,2022-01-03,28.7,19.4,0.0,23.7,42298.08
3,2022-01-04,30.4,21.7,3.6,27.9,49152.16
4,2022-01-05,28.4,21.5,2.2,20.6,35602.70


In [8]:
weather = weather.rename(columns={
    "time": "Date",
    "temperature_2m_max": "Max_Temp",
    "temperature_2m_min": "Min_Temp",
    "precipitation_sum": "Rainfall",
    "wind_speed_10m_max": "Max_Wind",
    "sunshine_duration": "Sunshine_Duration"
})

In [9]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Date               1461 non-null   str    
 1   Max_Temp           1461 non-null   float64
 2   Min_Temp           1461 non-null   float64
 3   Rainfall           1461 non-null   float64
 4   Max_Wind           1461 non-null   float64
 5   Sunshine_Duration  1461 non-null   float64
dtypes: float64(5), str(1)
memory usage: 82.9 KB


In [10]:
weather.describe()

,Max_Temp,Min_Temp,Rainfall,Max_Wind,Sunshine_Duration
count,1461.000000,1461.000000,1461.000000,1461.000000,1461.000000
mean,25.591444,15.630801,4.052841,15.777276,35982.839699
std,4.115436,4.374465,11.690928,4.143723,11513.549336
min,13.700000,5.200000,0.000000,6.400000,0.000000
25%,22.300000,11.900000,0.000000,12.800000,35530.690000
50%,25.900000,16.100000,0.300000,15.300000,38593.050000
75%,28.700000,19.300000,2.400000,18.300000,43200.000000
max,37.900000,25.000000,167.800000,41.800000,49314.090000


In [11]:
weather.head()

,Date,Max_Temp,Min_Temp,Rainfall,Max_Wind,Sunshine_Duration
0,2022-01-01,25.5,19.5,14.3,13.3,17634.44
1,2022-01-02,27.5,18.4,2.6,21.1,48883.74
2,2022-01-03,28.7,19.4,0.0,23.7,42298.08
3,2022-01-04,30.4,21.7,3.6,27.9,49152.16
4,2022-01-05,28.4,21.5,2.2,20.6,35602.70


In [18]:
weather.to_csv(
    "../data/external/weather.csv",
    index=False
)

In [12]:
weather.shape

(1461, 6)

In [13]:
weather.isnull().sum()

Date                 0
Max_Temp             0
Min_Temp             0
Rainfall             0
Max_Wind             0
Sunshine_Duration    0
dtype: int64

In [14]:
weather.describe()

,Max_Temp,Min_Temp,Rainfall,Max_Wind,Sunshine_Duration
count,1461.000000,1461.000000,1461.000000,1461.000000,1461.000000
mean,25.591444,15.630801,4.052841,15.777276,35982.839699
std,4.115436,4.374465,11.690928,4.143723,11513.549336
min,13.700000,5.200000,0.000000,6.400000,0.000000
25%,22.300000,11.900000,0.000000,12.800000,35530.690000
50%,25.900000,16.100000,0.300000,15.300000,38593.050000
75%,28.700000,19.300000,2.400000,18.300000,43200.000000
max,37.900000,25.000000,167.800000,41.800000,49314.090000


In [15]:
weather["Date"] = pd.to_datetime(weather["Date"])

In [16]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Date               1461 non-null   datetime64[us]
 1   Max_Temp           1461 non-null   float64       
 2   Min_Temp           1461 non-null   float64       
 3   Rainfall           1461 non-null   float64       
 4   Max_Wind           1461 non-null   float64       
 5   Sunshine_Duration  1461 non-null   float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 68.6 KB


In [17]:
weather["Date"].min(), weather["Date"].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [18]:
weather.duplicated().sum()

np.int64(0)

### EDA FOR Weather dataset

#### How does temperature change over 4 years in Brisbane?

In [20]:
fig = px.line(
    weather,
    x="Date",
    y="Max_Temp",
    title="Daily Maximum Temperature in Brisbane"
)

fig.show()

In [22]:
fig = px.line(
    weather,
    x="Date",
    y="Min_Temp",
    title="Daily Minimum Temperature in Brisbane"
)

fig.show()

#### Rainfall in mm

In [25]:
fig = px.line(
    weather,
    x="Date",
    y="Rainfall",
    title="Daily Rainfall"
)

fig.show()

#### Temperature distribution

In [27]:
fig = px.histogram(
    weather,
    x="Max_Temp",
    nbins=30,
    title="Distribution of Maximum Temperature"
)

fig.show()

### Can every traffic observation be associated with the weather conditions on that day?

In [28]:
df["Date"] = pd.to_datetime(df["Date"])
weather["Date"] = pd.to_datetime(weather["Date"])

In [31]:
master_weather = df.merge(
    weather,
    on="Date",
    how="left"
)

In [36]:
master_weather.shape #should have same rows as df and 5 extra cols of weather_df

(57752, 20)

In [37]:
df.shape

(57752, 15)

In [38]:
master_weather.head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date,Max_Temp,Min_Temp,Rainfall,Max_Wind,Sunshine_Duration
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013,19.4,10.5,0.0,14.9,36000.00
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013,13.8,10.7,4.9,14.3,0.00
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013,13.7,10.2,17.0,10.9,0.00
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013,18.1,11.3,6.0,15.0,17139.52
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013,21.8,12.7,0.0,27.1,36000.00


In [39]:
master_weather[
    [
        "Max_Temp",
        "Min_Temp",
        "Rainfall",
        "Max_Wind",
        "Sunshine_Duration"
    ]
].isnull().sum()

Max_Temp             0
Min_Temp             0
Rainfall             0
Max_Wind             0
Sunshine_Duration    0
dtype: int64

### Advanced feature engineering

#### Feature 1 - Rainy day

In [40]:
master_weather["Rainy_Day"] = np.where(
    master_weather["Rainfall"] > 0,
    1,
    0
)

#### Feature 2 - Heavy rain

In [42]:
#rainfall > 10mm
master_weather["Heavy_Rain"] = np.where(
    master_weather["Rainfall"] >= 10,
    1,
    0
)

#### Feature 3 - Hot day

In [44]:
#temp > 30 degrees celsius
master_weather["Hot_Day"] = np.where(
    master_weather["Max_Temp"] >= 30,
    1,
    0
)

#### Feature 4 - Cold mornings

In [45]:
master_weather["Cold_Morning"] = np.where(
    master_weather["Min_Temp"] <= 15,
    1,
    0
)

#### Feature 5 - Comfortable weather

In [46]:
master_weather["Comfortable_Weather"] = np.where(
    (
        (master_weather["Max_Temp"] >= 20)
        &
        (master_weather["Max_Temp"] <= 28)
        &
        (master_weather["Rainfall"] == 0)
    ),
    1,
    0
)

In [47]:
master_weather[
    [
        "Rainfall",
        "Max_Temp",
        "Min_Temp",
        "Rainy_Day",
        "Heavy_Rain",
        "Hot_Day",
        "Cold_Morning",
        "Comfortable_Weather"
    ]
].head(10)

,Rainfall,Max_Temp,Min_Temp,Rainy_Day,Heavy_Rain,Hot_Day,Cold_Morning,Comfortable_Weather
0,0.0,19.4,10.5,0,0,0,1,0
1,4.9,13.8,10.7,1,0,0,1,0
2,17.0,13.7,10.2,1,1,0,1,0
3,6.0,18.1,11.3,1,0,0,1,0
4,0.0,21.8,12.7,0,0,0,1,1
5,0.0,18.0,9.7,0,0,0,1,0
6,0.0,17.6,6.9,0,0,0,1,0
7,0.0,18.7,6.4,0,0,0,1,0
8,0.3,19.5,7.0,1,0,0,1,0
9,13.4,18.7,7.3,1,1,0,1,0


#### Daily temperature range

In [48]:
master_weather["Temp_Range"] = (
    master_weather["Max_Temp"]
    -
    master_weather["Min_Temp"]
)

In [49]:
master_weather["Windy_Day"] = (
    master_weather["Max_Wind"] >= 25
).astype(int)

In [51]:
#sunshine hours
master_weather["Sunshine_Hours"] = (
    master_weather["Sunshine_Duration"] / 3600
)

### Time series features

#### Previous day traffic

In [52]:
master_weather = master_weather.sort_values(
    ["Site_ID", "Mode", "Date"]
)

In [53]:
master_weather["Previous_Day_Count"] = (
    master_weather
        .groupby(["Site_ID", "Mode"])["Count"]
        .shift(1)
)

#### 7 day rolling avg

In [54]:
master_weather["Rolling_7Day_Avg"] = (
    master_weather
        .groupby(["Site_ID", "Mode"])["Count"]
        .transform(
            lambda x:
            x.rolling(
                window=7,
                min_periods=1
            ).mean()
        )
)

#### Rolling standard deviation

In [55]:
master_weather["Rolling_7Day_STD"] = (
    master_weather
        .groupby(["Site_ID", "Mode"])["Count"]
        .transform(
            lambda x:
            x.rolling(
                7,
                min_periods=1
            ).std()
        )
)

In [57]:
#is Holiday season?
master_weather["Holiday_Season"] = (
    master_weather["Month"].isin([12, 1])
).astype(int)

#### Cyclical Encoding

In [58]:
master_weather["Month_Sin"] = np.sin(
    2 * np.pi * master_weather["Month"] / 12
)

master_weather["Month_Cos"] = np.cos(
    2 * np.pi * master_weather["Month"] / 12
)

In [61]:
master_weather["DayOfWeek_Sin"] = np.sin(
    2 * np.pi * master_weather["Day"] / 7
)

master_weather["DayOfWeek_Cos"] = np.cos(
    2 * np.pi * master_weather["Day"] / 7
)

In [62]:
master_weather.head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date,Max_Temp,Min_Temp,Rainfall,Max_Wind,Sunshine_Duration,Rainy_Day,Heavy_Rain,Hot_Day,Cold_Morning,Comfortable_Weather,Temp_Range,Windy_Day,Sunshine_Hours,Previous_Day_Count,Rolling_7Day_Avg,Rolling_7Day_STD,Holiday_Season,Month_Sin,Month_Cos,DayOfWeek_Sin,DayOfWeek_Cos
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013,19.4,10.5,0.0,14.9,36000.00,0,0,0,1,0,8.9,0,10.000000,NaN,1246.00,NaN,0,-0.5,-0.866025,4.338837e-01,-0.900969
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013,13.8,10.7,4.9,14.3,0.00,1,0,0,1,0,3.1,0,0.000000,1246.0,1409.50,231.223917,0,-0.5,-0.866025,-4.338837e-01,-0.900969
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013,13.7,10.2,17.0,10.9,0.00,1,1,0,1,0,3.5,0,0.000000,1573.0,1143.00,489.692761,0,-0.5,-0.866025,-9.749279e-01,-0.222521
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013,18.1,11.3,6.0,15.0,17139.52,1,0,0,1,0,6.8,0,4.760978,610.0,1194.75,413.011198,0,-0.5,-0.866025,-7.818315e-01,0.623490
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013,21.8,12.7,0.0,27.1,36000.00,0,0,0,1,1,9.1,1,10.000000,1350.0,1435.60,646.511639,0,-0.5,-0.866025,-2.449294e-16,1.000000


### Weather impact analysis

#### Does rainfall affect traffic?

In [63]:
rain_effect = (
    master_weather
    .groupby(["Rainy_Day", "Mode"])["Count"]
    .mean()
    .reset_index()
)

rain_effect["Rainy_Day"] = rain_effect["Rainy_Day"].map({
    0: "No Rain",
    1: "Rain"
})

In [64]:
fig = px.bar(
    rain_effect,
    x="Rainy_Day",
    y="Count",
    color="Mode",
    barmode="group",
    text_auto=".0f",
    title="Average Daily Traffic on Rainy vs Non-Rainy Days"
)

fig.show()

#### Temperature vs Traffic

In [65]:
master_weather["Temp_Band"] = pd.cut(
    master_weather["Max_Temp"],
    bins=[10,20,25,30,35,40],
    labels=[
        "10-20°C",
        "20-25°C",
        "25-30°C",
        "30-35°C",
        "35°C+"
    ]
)

In [66]:
temp_effect = (
    master_weather
    .groupby(["Temp_Band","Mode"])["Count"]
    .mean()
    .reset_index()
)

In [67]:
fig = px.bar(
    temp_effect,
    x="Temp_Band",
    y="Count",
    color="Mode",
    barmode="group",
    title="Average Traffic by Temperature Band"
)

fig.show()

#### Does wind affect mobility?

In [69]:
wind_effect = (
    master_weather
    .groupby(["Windy_Day", "Mode"])["Count"]
    .mean()
    .reset_index()
)

wind_effect["Windy_Day"] = wind_effect["Windy_Day"].map({
    0: "Normal Wind",
    1: "Windy Day"
})

wind_effect

,Windy_Day,Mode,Count
0,Normal Wind,Cyclist,656.936586
1,Normal Wind,Pedestrian,1244.132477
2,Normal Wind,Scooter,482.454712
3,Windy Day,Cyclist,520.679612
4,Windy Day,Pedestrian,983.524487
5,Windy Day,Scooter,377.654762


In [70]:
fig = px.bar(
    wind_effect,
    x="Windy_Day",
    y="Count",
    color="Mode",
    barmode="group",
    text_auto=".0f",
    title="Average Daily Traffic on Windy vs Normal Wind Days"
)

fig.update_layout(
    xaxis_title="Weather Condition",
    yaxis_title="Average Daily Traffic"
)

fig.show()

#### Correlation b/w weather variables and traffic

In [71]:
master_weather["Sunshine_Hours"] = (
    master_weather["Sunshine_Duration"] / 3600
)

In [72]:
weather_corr = master_weather[
    [
        "Count",
        "Max_Temp",
        "Min_Temp",
        "Rainfall",
        "Max_Wind",
        "Sunshine_Hours"
    ]
].corr()

In [73]:
fig = px.imshow(
    weather_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    aspect="auto",
    title="Correlation Between Weather Variables and Traffic"
)

fig.show()

#### Monthly weather vs traffic

In [74]:
monthly_weather = (
    master_weather
    .groupby("Month")
    .agg(
        Avg_Traffic=("Count", "mean"),
        Avg_Max_Temp=("Max_Temp", "mean"),
        Avg_Rainfall=("Rainfall", "mean")
    )
    .reset_index()
)

monthly_weather

,Month,Avg_Traffic,Avg_Max_Temp,Avg_Rainfall
0,1,826.520798,29.120948,5.103022
1,2,875.338976,29.226888,7.767187
2,3,831.604475,28.147323,7.981762
3,4,930.212518,25.739218,2.497297
4,5,872.702990,22.655308,4.472868
5,6,854.303992,20.943067,0.689685
6,7,834.929188,20.416209,1.712518
7,8,896.482003,22.559602,1.481339
8,9,915.617082,24.759262,2.540651
9,10,897.929556,26.792826,3.535610


In [75]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])

In [76]:
fig.add_trace(
    go.Scatter(
        x=monthly_weather["Month"],
        y=monthly_weather["Avg_Traffic"],
        name="Average Traffic",
        mode="lines+markers"
    ),
    secondary_y=False
)


In [77]:
fig.add_trace(
    go.Scatter(
        x=monthly_weather["Month"],
        y=monthly_weather["Avg_Max_Temp"],
        name="Average Max Temperature",
        mode="lines+markers"
    ),
    secondary_y=True
)

In [78]:
fig.update_layout(
    title="Monthly Average Traffic vs Maximum Temperature"
)

fig.update_xaxes(title_text="Month")

fig.update_yaxes(
    title_text="Average Daily Traffic",
    secondary_y=False
)

fig.update_yaxes(
    title_text="Average Maximum Temperature (°C)",
    secondary_y=True
)

fig.show()

#### w.r.t rainfall

In [79]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

In [80]:
fig.add_trace(
    go.Scatter(
        x=monthly_weather["Month"],
        y=monthly_weather["Avg_Traffic"],
        name="Average Traffic",
        mode="lines+markers"
    ),
    secondary_y=False
)

In [81]:
fig.add_trace(
    go.Bar(
        x=monthly_weather["Month"],
        y=monthly_weather["Avg_Rainfall"],
        name="Average Rainfall",
        opacity=0.5
    ),
    secondary_y=True
)

In [82]:
fig.update_layout(
    title="Monthly Average Traffic vs Rainfall"
)

fig.update_xaxes(title_text="Month")

fig.update_yaxes(
    title_text="Average Daily Traffic",
    secondary_y=False
)

fig.update_yaxes(
    title_text="Average Rainfall (mm)",
    secondary_y=True
)

fig.show()

### Time series features

In [83]:
master_weather = master_weather.sort_values(
    ["Site_ID", "Mode", "Date"]
).reset_index(drop=True)

#### Previous day traffic - Lag 1

In [84]:
master_weather["Lag_1"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .shift(1)
)

#### Previous week traffic - Lag 7

In [85]:
master_weather["Lag_7"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .shift(7)
)

#### Previous month traffic - Lag 30

In [86]:
master_weather["Lag_30"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .shift(30)
)

#### 7 day rolling mean

In [87]:
master_weather["Rolling_Mean_7"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=1
        ).mean()
    )
)

#### 30 day rolling mean

In [88]:
master_weather["Rolling_Mean_30"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=30,
            min_periods=1
        ).mean()
    )
)

#### 7 day rolling std deviation

In [89]:
master_weather["Rolling_STD_7"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=2
        ).std()
    )
)

#### Exponential moving average

In [90]:
master_weather["EMA_7"] = (
    master_weather
    .groupby(["Site_ID", "Mode"])["Count"]
    .transform(
        lambda x: x.shift(1).ewm(
            span=7,
            adjust=False
        ).mean()
    )
)

#### Cyclical Encoding

In [91]:
master_weather["Month_Sin"] = np.sin(
    2 * np.pi * master_weather["Month"] / 12
)

master_weather["Month_Cos"] = np.cos(
    2 * np.pi * master_weather["Month"] / 12
)

In [93]:
master_weather["Day_Sin"] = np.sin(
    2 * np.pi * master_weather["Day"] / 7
)

master_weather["Day_Cos"] = np.cos(
    2 * np.pi * master_weather["Day"] / 7
)

In [94]:
master_weather[
    [
        "Date",
        "Site_ID",
        "Mode",
        "Count",
        "Lag_1",
        "Lag_7",
        "Lag_30",
        "Rolling_Mean_7",
        "Rolling_Mean_30",
        "Rolling_STD_7",
        "EMA_7"
    ]
].head(15)

,Date,Site_ID,Mode,Count,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7,EMA_7
0,2022-07-03,A001,Cyclist,1246.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-07-04,A001,Cyclist,1573.0,1246.0,NaN,NaN,1246.000000,1246.000000,NaN,1246.000000
2,2022-07-05,A001,Cyclist,610.0,1573.0,NaN,NaN,1409.500000,1409.500000,231.223917,1327.750000
3,2022-07-06,A001,Cyclist,1350.0,610.0,NaN,NaN,1143.000000,1143.000000,489.692761,1148.312500
4,2022-07-07,A001,Cyclist,2399.0,1350.0,NaN,NaN,1194.750000,1194.750000,413.011198,1198.734375
5,2022-07-08,A001,Cyclist,2333.0,2399.0,NaN,NaN,1435.600000,1435.600000,646.511639,1498.800781
6,2022-07-09,A001,Cyclist,2051.0,2333.0,NaN,NaN,1585.166667,1585.166667,684.545810,1707.350586
7,2022-07-10,A001,Cyclist,2138.0,2051.0,1246.0,NaN,1651.714286,1651.714286,649.232294,1793.262939
8,2022-07-11,A001,Cyclist,2093.0,2138.0,1573.0,NaN,1779.142857,1712.500000,643.845072,1879.447205
9,2022-07-12,A001,Cyclist,2086.0,2093.0,610.0,NaN,1853.428571,1754.777778,646.091030,1932.835403


In [95]:
master_weather.head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date,Max_Temp,Min_Temp,Rainfall,Max_Wind,Sunshine_Duration,Rainy_Day,Heavy_Rain,Hot_Day,Cold_Morning,Comfortable_Weather,Temp_Range,Windy_Day,Sunshine_Hours,Previous_Day_Count,Rolling_7Day_Avg,Rolling_7Day_STD,Holiday_Season,Month_Sin,Month_Cos,DayOfWeek_Sin,DayOfWeek_Cos,Temp_Band,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7,EMA_7,Day_Sin,Day_Cos
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013,19.4,10.5,0.0,14.9,36000.00,0,0,0,1,0,8.9,0,10.000000,NaN,1246.00,NaN,0,-0.5,-0.866025,4.338837e-01,-0.900969,10-20°C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.338837e-01,-0.900969
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013,13.8,10.7,4.9,14.3,0.00,1,0,0,1,0,3.1,0,0.000000,1246.0,1409.50,231.223917,0,-0.5,-0.866025,-4.338837e-01,-0.900969,10-20°C,1246.0,NaN,NaN,1246.00,1246.00,NaN,1246.000000,-4.338837e-01,-0.900969
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013,13.7,10.2,17.0,10.9,0.00,1,1,0,1,0,3.5,0,0.000000,1573.0,1143.00,489.692761,0,-0.5,-0.866025,-9.749279e-01,-0.222521,10-20°C,1573.0,NaN,NaN,1409.50,1409.50,231.223917,1327.750000,-9.749279e-01,-0.222521
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013,18.1,11.3,6.0,15.0,17139.52,1,0,0,1,0,6.8,0,4.760978,610.0,1194.75,413.011198,0,-0.5,-0.866025,-7.818315e-01,0.623490,10-20°C,610.0,NaN,NaN,1143.00,1143.00,489.692761,1148.312500,-7.818315e-01,0.623490
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013,21.8,12.7,0.0,27.1,36000.00,0,0,0,1,1,9.1,1,10.000000,1350.0,1435.60,646.511639,0,-0.5,-0.866025,-2.449294e-16,1.000000,20-25°C,1350.0,NaN,NaN,1194.75,1194.75,413.011198,1198.734375,-2.449294e-16,1.000000


In [96]:
master_weather.columns

Index(['Date', 'Year', 'Month', 'Month_Name', 'Day', 'Day_of_Week', 'Weekday_Weekend', 'Site_ID', 'Site Name', 'Mode', 'Count', 'Latitude', 'Longitude', 'Type', 'Start_Date', 'Max_Temp', 'Min_Temp',
       'Rainfall', 'Max_Wind', 'Sunshine_Duration', 'Rainy_Day', 'Heavy_Rain', 'Hot_Day', 'Cold_Morning', 'Comfortable_Weather', 'Temp_Range', 'Windy_Day', 'Sunshine_Hours', 'Previous_Day_Count',
       'Rolling_7Day_Avg', 'Rolling_7Day_STD', 'Holiday_Season', 'Month_Sin', 'Month_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Temp_Band', 'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_Mean_7',
       'Rolling_Mean_30', 'Rolling_STD_7', 'EMA_7', 'Day_Sin', 'Day_Cos'],
      dtype='str')

In [97]:
master_weather.drop(
    columns=[
        "Previous_Day_Count",
        "Rolling_7Day_Avg",
        "Rolling_7Day_STD",
        "DayOfWeek_Sin",
        "DayOfWeek_Cos",
        "Sunshine_Duration",
        "Temp_Band"
    ],
    inplace=True
)

In [98]:
master_weather.head(10)

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date,Max_Temp,Min_Temp,Rainfall,Max_Wind,Rainy_Day,Heavy_Rain,Hot_Day,Cold_Morning,Comfortable_Weather,Temp_Range,Windy_Day,Sunshine_Hours,Holiday_Season,Month_Sin,Month_Cos,Lag_1,Lag_7,Lag_30,Rolling_Mean_7,Rolling_Mean_30,Rolling_STD_7,EMA_7,Day_Sin,Day_Cos
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013,19.4,10.5,0.0,14.9,0,0,0,1,0,8.9,0,10.000000,0,-0.5,-0.866025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.338837e-01,-0.900969
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013,13.8,10.7,4.9,14.3,1,0,0,1,0,3.1,0,0.000000,0,-0.5,-0.866025,1246.0,NaN,NaN,1246.000000,1246.000000,NaN,1246.000000,-4.338837e-01,-0.900969
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013,13.7,10.2,17.0,10.9,1,1,0,1,0,3.5,0,0.000000,0,-0.5,-0.866025,1573.0,NaN,NaN,1409.500000,1409.500000,231.223917,1327.750000,-9.749279e-01,-0.222521
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013,18.1,11.3,6.0,15.0,1,0,0,1,0,6.8,0,4.760978,0,-0.5,-0.866025,610.0,NaN,NaN,1143.000000,1143.000000,489.692761,1148.312500,-7.818315e-01,0.623490
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013,21.8,12.7,0.0,27.1,0,0,0,1,1,9.1,1,10.000000,0,-0.5,-0.866025,1350.0,NaN,NaN,1194.750000,1194.750000,413.011198,1198.734375,-2.449294e-16,1.000000
5,2022-07-08,2022,7,July,8,Friday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2333.0,-27.478042,153.00035,Automatic,2013,18.0,9.7,0.0,22.7,0,0,0,1,0,8.3,0,10.015189,0,-0.5,-0.866025,2399.0,NaN,NaN,1435.600000,1435.600000,646.511639,1498.800781,7.818315e-01,0.623490
6,2022-07-09,2022,7,July,9,Saturday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2051.0,-27.478042,153.00035,Automatic,2013,17.6,6.9,0.0,18.9,0,0,0,1,0,10.7,0,10.013894,0,-0.5,-0.866025,2333.0,NaN,NaN,1585.166667,1585.166667,684.545810,1707.350586,9.749279e-01,-0.222521
7,2022-07-10,2022,7,July,10,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2138.0,-27.478042,153.00035,Automatic,2013,18.7,6.4,0.0,13.0,0,0,0,1,0,12.3,0,10.012414,0,-0.5,-0.866025,2051.0,1246.0,NaN,1651.714286,1651.714286,649.232294,1793.262939,4.338837e-01,-0.900969
8,2022-07-11,2022,7,July,11,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2093.0,-27.478042,153.00035,Automatic,2013,19.5,7.0,0.3,16.6,1,0,0,1,0,12.5,0,9.987869,0,-0.5,-0.866025,2138.0,1573.0,NaN,1779.142857,1712.500000,643.845072,1879.447205,-4.338837e-01,-0.900969
9,2022-07-12,2022,7,July,12,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2086.0,-27.478042,153.00035,Automatic,2013,18.7,7.3,13.4,8.0,1,1,0,1,0,11.4,0,6.000000,0,-0.5,-0.866025,2093.0,610.0,NaN,1853.428571,1754.777778,646.091030,1932.835403,-9.749279e-01,-0.222521


In [99]:
lag_columns = [
    "Lag_1",
    "Lag_7",
    "Lag_30",
    "Rolling_Mean_7",
    "Rolling_Mean_30",
    "Rolling_STD_7",
    "EMA_7"
]

master_weather[lag_columns].isnull().sum()

Lag_1                69
Lag_7               483
Lag_30             2070
Rolling_Mean_7       69
Rolling_Mean_30      69
Rolling_STD_7       138
EMA_7                69
dtype: int64

In [100]:
master_weather.to_parquet(
    "../data/processed/master_weather.parquet",
    index=False
)

In [101]:
master_weather.to_csv(
    "../data/processed/master_weather.csv",
    index=False
)

#### Summary
1. Downloaded historical Brisbane weather data using the Open-Meteo API.
2. Cleaned and validated the weather dataset.
3. Merged weather information with the traffic monitoring dataset.
4. Analysed how rainfall, temperature and wind influence active mobility.
5. Engineered weather-related features (e.g. Rainy_Day, Hot_Day, Comfortable_Weather).
6. Created advanced time-series features, including lag variables, rolling statistics and cyclical encodings, to improve forecasting capability.
7. Produced the final modelling dataset for machine learning.

#### Key Takeaways
1. Weather has a measurable influence on pedestrian, cyclist and scooter activity.
2. Traffic exhibits strong temporal dependencies, making lag and rolling features valuable predictors.
3. Cyclical encoding enables machine learning models to better represent recurring weekly and yearly patterns.
4. The enhanced dataset now combines spatial, temporal, weather and historical traffic information, providing a comprehensive foundation for predictive modelling.